In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_DAGBagM"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("nij root:" , nij_root)
print("Output directory:", output_dir)

nij root: /dcs/23/u2200504/thesis/recidivism-causal/data/processed
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/NIJ/graphs_DAGBagM


In [3]:
#assigns node types. 'c' for continuous, 'b' for binary
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [4]:
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    df_clean = encode_mixed_df(df_clean)
    #infer node types on transformed df
    node_type = infer_type(df_clean)

    print("Columns used in dagbagM:", df_clean.columns.tolist())
    print("dtypes:", df_clean.dtypes)

    #start = time.time()
    with (ro.default_converter + pandas2ri.converter).context():
        #Python ->R
        Y_r = conversion.py2rpy(df_clean)

        ro.globalenv["Y"] = Y_r
        ro.globalenv["node_type"] = node_type
        ro.globalenv["seed"] = seed
        ro.r('''
        set.seed(seed)
        Y_df <- as.data.frame(Y)

        # Convert to numeric matrix for hc
        Y_mat <- as.matrix(Y_df)
        
        temp <- dagbagM::hc(
          Y = Y_mat,
          nodeType = node_type,
          whiteList = NULL,
          blackList = NULL,
          tol = 1e-6,
          standardize = FALSE,
          maxStep = 1000,
          restart = 10,
          verbose = FALSE
        )
        adj_mat <- temp$adjacency
        col_names <- colnames(Y_mat)
        ''')
        #R -> Python
        adjacency = conversion.rpy2py(ro.r('adj_mat'))
        col_names= list(ro.r('col_names'))

    #end = time.time()
    #print(f"DAGBagM took {(end - start)/60:.2f} minutes")
    return np.asarray(adjacency), col_names #list(df_clean.columns) #returns labels.

In [5]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [6]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [10]:
csv_path = output_dir/ "NIJ_graph_DAGBagM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        adj, _ = run_dagbagm(df_subset, seed=1)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

#score = incompatibility_score(A, k=5, n_subsets=50, seed=42)
#print("Approx. incompatibility score:", score)


In [12]:
incompat_score = 7.44

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))

#goodness(incompat_score, 5)

In [8]:
score_seed7 = incompatibility_score(A, k=5, n_subsets=50, seed=7)
print("Approx. incompatibility score:", score_seed7)
goodness(score_seed7, 5)

Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_28-32', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Drug_high']
dtypes: Race_BLACK                            int8
Age_at_Release_18-22                  int8
Age_at_Release_28-32                  int8
Prior_Arrest_Episodes_Felony_high    int64
Prior_Arrest_Episodes_Drug_high      int64
dtype: object
DAGBagM took 0.05 minutes
Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_23-27', 'Prior_Arrest_Episodes_Property_high', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive']
dtypes: Race_BLACK                                int8
Age_at_Release_23-27                      int8
Prior_Arrest_Episodes_Property_high      int64
Avg_Days_per_DrugTest                  float64
DrugTests_Cocaine_Positive             float64
dtype: object
DAGBagM took 0.06 minutes
Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_38-42', 'Prior_Conviction_Episodes_Felony_high', 'DrugTe

In [9]:
score_seed12 = incompatibility_score(A, k=5, n_subsets=50, seed=12)
print("Approx. incompatibility score:", score_seed12)
goodness(score_seed12, 5)

Columns used in dagbagM: ['Age_at_Release_28-32', 'Age_at_Release_48 or older', 'Prior_Arrest_Episodes_Property_high', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest']
dtypes: Age_at_Release_28-32                      int8
Age_at_Release_48 or older                int8
Prior_Arrest_Episodes_Property_high      int64
Jobs_Per_Year                          float64
Avg_Days_per_DrugTest                  float64
dtype: object
DAGBagM took 0.06 minutes
Columns used in dagbagM: ['Gender_F', 'Age_at_Release_48 or older', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Jobs_Per_Year']
dtypes: Gender_F                                  int8
Age_at_Release_48 or older                int8
Prior_Arrest_Episodes_Property_high      int64
Prior_Arrest_Episodes_Drug_high          int64
Jobs_Per_Year                          float64
dtype: object
DAGBagM took 0.03 minutes
Columns used in dagbagM: ['Gender_F', 'Age_at_Release_43-47', 'Percent_Days_Employed', 'DrugTests_Cocaine_Positive

In [5]:
seeds = [42, 7, 12]
incompat_scores = [7.44, 8.28, 6.78]
disagree = [37.2, 41.4, 33.9]

print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 0.6138403701289119
standard dev of disagreement percentage: 3.069201850644561


In [7]:
csv_path = nij_root / "NIJ_lean_compact_onehot.csv"
df_full = pd.read_csv(csv_path)
node_labels = list(df_full.columns)
d = len(node_labels)

def bootstrap_edge_stability(df, B=20, seed=0):
    rng = np.random.default_rng(seed)
    edge_counts = np.zeros((d, d), dtype=int)

    for b in range(B):
        #sample B rows with replacement from original learning data
        idx = rng.integers(low=0, high=len(df), size=len(df))
        df_boot = df.iloc[idx, :]

        #run DAGSLAM on each bootstrap sample
        W_est, _ = run_dagbagm(df_boot, seed=1)           # shape (d, d), aligned with columns
        W_bin = (np.asarray(W_est) != 0).astype(int)

        #accumulate edge counts
        edge_counts += W_bin

    #convert to percentage appearances
    edge_freq = edge_counts / B
    return edge_freq
    
edge_freq_dagbagm = bootstrap_edge_stability(df_full, B=20, seed=42)

#save result to CSV
edge_freq_df = pd.DataFrame(edge_freq_dagbagm, index=node_labels, columns=node_labels)
edge_freq_df.to_csv(output_dir / "NIJ_DAGBagM_edge_stability.csv")
print("Number of edges with freq >= 0.5:",
      np.sum(edge_freq_dagbagm >= 0.5))
print("Number of edges with freq >= 0.8:",
      np.sum(edge_freq_dagbagm >= 0.8))


Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '
R callback write-console: In addition:   
R callback write-console: There were 20 warnings (use warnings() to see them)  
R callback write-console: 
  
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '


Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

R callback write-console: In addition:   
R callback write-console: There were 20 warnings (use warnings() to see them)  
R callback write-console: 
  
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '


Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

...

Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '
R callback write-console: In addition:   
R callback write-console: There were 50 or more warnings (use warnings() to see the first 50)  
R callback write-console: 
  
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '


Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

R callback write-console: In addition:   
R callback write-console: There were 46 warnings (use warnings() to see them)  
R callback write-console: 
  


Number of edges with freq >= 0.5: 128
Number of edges with freq >= 0.8: 80


In [13]:
csv_path = output_dir/ "NIJ_graph_DAGBagM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()
subset_sizes=[5,10,15]
incompat_scores = []
disagree = []
for size in subset_sizes:
    score = incompatibility_score(A, k = size, n_subsets=50, seed=42)
    incompat_scores.append(score)
    disagree.append(goodness(score, size))
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

Columns used in dagbagM: ['Gender_F', 'Age_at_Release_23-27', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Recidivism_Within_3years']
dtypes: Gender_F                      int8
Age_at_Release_23-27          int8
Age_at_Release_43-47          int8
Age_at_Release_48 or older    int8
Recidivism_Within_3years      int8
dtype: object
Columns used in dagbagM: ['Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_43-47', 'DrugTests_Cocaine_Positive']
dtypes: Age_at_Release_18-22             int8
Age_at_Release_23-27             int8
Age_at_Release_28-32             int8
Age_at_Release_43-47             int8
DrugTests_Cocaine_Positive    float64
dtype: object
Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Prior_Conviction_Episodes_Felony_high', 'Delinquency_Reports_high']
dtypes: Gender_F                                  int8
Race_BLACK                                int8
Age_at_Release_18-22                      int8
P

R callback write-console: In addition:   
R callback write-console: There were 14 warnings (use warnings() to see them)  
R callback write-console: 
  


Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Recidivism_Within_3years']
dtypes: Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 or older                  int8
Gang_Affiliated                             int8
Prior_Arrest_Episodes_Felony_high          int64
Prior_Arrest_Episodes_Property_high        int64
Prior_Arrest_Episodes_Drug_high            int64
Prior_Conviction_Episo

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 2:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  algorithm did not converge
  
R callback write-console: 3:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 4:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   


Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_23-27', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Race_BLACK                                  int8
Age_at_Release_23-27                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 or older                  int8
Prior_Arrest_Episodes_Felony_high          int64
Prior_Arrest_Episodes_Property_high        int64
Prior_Conviction_Episodes_Felony_high      int64
Percent_Days_Employed                    float64
Jobs_Per_Year                            float64
Avg_Days_per_Dr

...

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 2:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  algorithm did not converge
  
R callback write-console: 3:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 4:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   


Columns used in dagbagM: ['Race_BLACK', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'Supervision_Risk_Score_First']
dtypes: Race_BLACK                                  int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 or older                  int8
Gang_Affiliated                             int8
Prior_Arrest_Episodes_Felony_high          int64
Prior_Arrest_Episodes_Property_high        int64
Prior_Arrest_Episodes_Drug_high            int64
Prior_Conviction_Episodes_Felony_high      int64
Percent_Days_Empl

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 2:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  algorithm did not converge
  
R callback write-console: 3:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 4:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   


Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Gang_Affiliated                             int8
Prior_Arrest_Episodes_Felony_high          int64
Prior_Arrest_Episodes_Property_high        int64
Prior_Arrest_Episodes_Drug_high            int64
Prior_Conviction_Episodes_Felony_high      int64
Percent_Days_Employed                    float64
Jobs_Per_Year                            float64
Avg_Days_per_DrugTest    

TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

In [14]:
incompat_scores

[7.44, 28.62, 52.48]

In [16]:
disagree

[None, None, None]